In [2]:
%%HTML
<style>
    body {
        --vscode-font-family: "Fira Sans"
    }
</style>

# Python Concurrency: Threading Tutorial

Threading in Python is primarily used for **I/O-bound tasks** (like web scraping, disk I/O, or network requests). Due to the **Global Interpreter Lock (GIL)**, only one thread can execute Python bytecode at a time, so threading does not provide true parallelism for CPU-bound tasks (use `multiprocessing` for that).

## 1. Thread Scheduling and the GIL

### Who schedules the threads?
Python threads are **real OS-level threads** (POSIX threads on Linux/macOS, Windows threads on Windows). This means they are managed and scheduled by the **Operating System kernel**, not by the Python language runtime. The OS decides which thread runs on which CPU core and for how long.

### What is the GIL?
The **Global Interpreter Lock (GIL)** is a mutex (or a lock) that protects access to Python objects, preventing multiple threads from executing Python bytecodes at once. 

### Why do we have it?
The GIL was implemented to make CPython's memory management (which is not thread-safe) easier to implement and to simplify the integration of C extensions. Without it, every object would need its own lock, which would be extremely slow for single-threaded programs.

### The Impact: Concurrency vs. Parallelism
- **Parallelism** (True simultaneous execution) is achieved for non-Python code (like I/O operations or CPU-intensive C extensions like NumPy) because they can release the GIL.
- **Concurrency** (Interleaved execution) is what happens for pure Python code. Even with multiple cores, only one thread can hold the GIL and execute Python code at any given moment. The OS switches between threads, giving the illusion of simultaneous execution.

## 2. Concurrency Primitives

Synchronization primitives are tools to manage access to shared resources and coordinate thread execution.

### threading.Lock
- **Use Case**: Mutual exclusion. Ensure only one thread accesses a shared resource (like a counter or a file) at a time.
- **Mechanism**: `acquire()` to lock, `release()` to unlock. Using a context manager (`with lock:`) is recommended.

In [4]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:
            counter += 1

threads = [threading.Thread(target=increment) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Final counter: {counter}")

Final counter: 200000


### threading.RLock (Re-entrant Lock)
- **Use Case**: When a thread needs to acquire the same lock multiple times (e.g., in recursive functions or nested method calls).
- **Mechanism**: A standard `Lock` would dead lock if the same thread tried to acquire it twice. `RLock` keeps track of the owner and recursion level.

In [5]:
rlock = threading.RLock()

def outer():
    with rlock:
        print("Acquired in outer")
        inner()

def inner():
    with rlock:
        print("Acquired in inner (no deadlock!)")

threading.Thread(target=outer).start()

Acquired in outer
Acquired in inner (no deadlock!)


### threading.Semaphore
- **Use Case**: Controlling access to a resource pool with a limited capacity (e.g., limiting concurrent database connections).
- **Mechanism**: Maintains a counter. `acquire()` decrements; `release()` increments. Blocks if counter is zero.

In [6]:
import time

# Limit to 2 concurrent threads
semaphore = threading.Semaphore(2)

def task(id):
    with semaphore:
        print(f"Thread {id} entering")
        time.sleep(1)
        print(f"Thread {id} leaving")

for i in range(5):
    threading.Thread(target=task, args=(i,)).start()

Thread 0 entering
Thread 1 entering


Thread 0 leavingThread 1 leaving

Thread 2 entering
Thread 3 entering
Thread 3 leaving
Thread 4 entering
Thread 2 leaving
Thread 4 leaving


### threading.Event
- **Use Case**: Simple signaling. One thread signals an event, and other threads wait for it (e.g., waiting for initialization to finish).
- **Mechanism**: Binary flag. `set()` to true, `clear()` to false, `wait()` blocks until true.

In [ ]:

import threading
import time

event = threading.Event()

def worker():
    print("Worker waiting for signal...")
    event.wait()
    print("Worker received signal, starting work!")

threading.Thread(target=worker).start()
time.sleep(2)
print("Main thread sending signal...")
event.set()

Worker waiting for signal...
Main thread sending signal...
Worker received signal, starting work!


### threading.Condition
- **Use Case**: Complex coordination where threads wait for a specific state change (e.g., Producer-Consumer where Consumer waits for a non-empty buffer).
- **Mechanism**: Combines a lock with a notification mechanism (`wait()`, `notify()`, `notify_all()`).

In [ ]:
import threading

condition = threading.Condition()
items = []

def consumer():
    with condition:
        while not items:
            print("Consumer waiting...")
            condition.wait()
        print(f"Consumer got: {items.pop()}")

def producer():
    time.sleep(1)
    with condition:
        items.append("Data")
        print("Producer notified!")
        condition.notify()

threading.Thread(target=consumer).start()
threading.Thread(target=producer).start()

## 3. Advanced Coordination & Management

Beyond simple primitives, Python provides tools for specific coordination patterns.

### threading.Barrier
- **Use Case**: Coordination of a fixed number of threads that must wait for each other to reach a certain point before any can proceed (e.g., parallel initialization).
- **Mechanism**: Set with a `parties` count. Threads call `wait()` and block until the count is reached.

In [11]:
barrier = threading.Barrier(3)

def synchronized_task(id):
    print(f"Thread {id} reaching barrier...")
    barrier.wait()
    print(f"Thread {id} passing barrier!")

for i in range(3):
    threading.Thread(target=synchronized_task, args=(i,)).start()

Thread 0 reaching barrier...
Thread 1 reaching barrier...
Thread 2 reaching barrier...
Thread 2 passing barrier!
Thread 0 passing barrier!
Thread 1 passing barrier!


### threading.Timer
- **Use Case**: Delayed execution of a function (e.g., a timeout or a scheduled background task).
- **Mechanism**: Subclass of `Thread`. `start()` begins the delay; `cancel()` can stop it before execution.

In [ ]:
def delayed_function():
    print("Timer executed!")

t = threading.Timer(2.0, delayed_function)
t.start()
print("Timer started (2s delay)...")

### threading.local
- **Use Case**: Storing data that is **private to each thread**. Each thread can refer to the same global object but will see its own unique values (e.g., database connections per thread).
- **Mechanism**: An object whose attributes are thread-local.

In [ ]:
my_data = threading.local()

def show_data():
    try:
        val = my_data.value
    except AttributeError:
        val = "None"
    print(f"{threading.current_thread().name}: {val}")

def worker(value):
    my_data.value = value
    show_data()

threading.Thread(target=worker, args=('A',), name='Thread-1').start()
threading.Thread(target=worker, args=('B',), name='Thread-2').start()

## 4. Inter-Thread Communication

Threads need ways to safely exchange data.

### queue.Queue
- **Use Case**: The **recommended way** to communicate. Ideal for producer-consumer patterns.
- **Mechanism**: Thread-safe FIFO queue. `put()` and `get()` handle locking internally, so you don't have to.

In [ ]:
from queue import Queue

q = Queue()

def producer():
    for i in range(5):
        q.put(i)
        print(f"Produced {i}")

def consumer():
    while True:
        item = q.get()
        if item is None: break
        print(f"Consumed {item}")
        q.task_done()

threading.Thread(target=producer).start()
c = threading.Thread(target=consumer)
c.daemon = True
c.start()

q.join() # Wait for all items to be processed
print("All work done!")

### Shared Variables with Locks
- **Use Case**: When you need to update a shared state directly rather than passing discrete messages.
- **Warning**: More error-prone than `Queue` as you must remember to lock every access.

In [ ]:
shared_data = {"status": "idle"}
data_lock = threading.Lock()

def update_status(new_status):
    with data_lock:
        shared_data["status"] = new_status
        print(f"Status updated to: {shared_data['status']}")

threading.Thread(target=update_status, args=("running",)).start()